<a href="https://colab.research.google.com/github/Pranayshukla0610/basic-python/blob/main/UBER_System_Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from abc import ABC, abstractmethod
import math
import uuid


# =========================
# LOCATION
# =========================
class Location:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distance(self, other):
        return math.sqrt((self.x - other.x)**2 + (self.y - other.y)**2)


# =========================
# USER (ABSTRACTION + INHERITANCE)
# =========================
class User:
    def __init__(self, name, location):
        self.id = str(uuid.uuid4())
        self.name = name
        self.location = location


class Driver(User):
    def __init__(self, name, location):
        super().__init__(name, location)
        self.available = True
        self.rating = 5.0

    def assign(self):
        self.available = False

    def complete_ride(self):
        self.available = True


class Rider(User):
    def request_ride(self, system, destination):
        return system.create_ride(self, destination)


# =========================
# RIDE (COMPOSITION)
# =========================
class Ride:
    def __init__(self, rider, driver, source, destination, price):
        self.id = str(uuid.uuid4())
        self.rider = rider
        self.driver = driver
        self.source = source
        self.destination = destination
        self.price = price
        self.status = "ONGOING"

    def complete(self):
        self.status = "COMPLETED"
        self.driver.complete_ride()


# =========================
# MATCHING STRATEGY (ABSTRACTION + POLYMORPHISM)
# =========================
class MatchingStrategy(ABC):

    @abstractmethod
    def find_driver(self, drivers, rider):
        pass


class NearestDriverStrategy(MatchingStrategy):

    def find_driver(self, drivers, rider):
        available = [d for d in drivers if d.available]
        if not available:
            return None

        return min(available, key=lambda d: d.location.distance(rider.location))


# =========================
# PRICING STRATEGY (POLYMORPHISM)
# =========================
class PricingStrategy(ABC):

    @abstractmethod
    def calculate_price(self, distance):
        pass


class NormalPricing(PricingStrategy):

    def calculate_price(self, distance):
        return distance * 10


class SurgePricing(PricingStrategy):

    def calculate_price(self, distance):
        return distance * 20


# =========================
# PAYMENT SYSTEM (ABSTRACTION)
# =========================
class Payment(ABC):

    @abstractmethod
    def pay(self, amount):
        pass


class UPI(Payment):

    def pay(self, amount):
        print(f"Paid ₹{amount} via UPI")


class Card(Payment):

    def pay(self, amount):
        print(f"Paid ₹{amount} via Card")


# =========================
# UBER SYSTEM (MANAGER CLASS)
# =========================
class UberSystem:

    def __init__(self, matching_strategy, pricing_strategy):
        self.drivers = []
        self.riders = []
        self.rides = []
        self.matching_strategy = matching_strategy
        self.pricing_strategy = pricing_strategy

    def add_driver(self, driver):
        self.drivers.append(driver)

    def add_rider(self, rider):
        self.riders.append(rider)

    def create_ride(self, rider, destination):

        driver = self.matching_strategy.find_driver(self.drivers, rider)

        if not driver:
            print("No drivers available")
            return None

        distance = rider.location.distance(destination)

        price = self.pricing_strategy.calculate_price(distance)

        driver.assign()

        ride = Ride(rider, driver, rider.location, destination, price)
        self.rides.append(ride)

        print(f"Ride started: {rider.name} → {driver.name} | Price: ₹{price}")

        return ride

    def complete_ride(self, ride, payment_method):
        ride.complete()
        payment_method.pay(ride.price)
        print("Ride completed")

In [4]:
system = UberSystem(NearestDriverStrategy(), NormalPricing())

d1 = Driver("Driver1", Location(0, 0))
d2 = Driver("Driver2", Location(10, 10))

system.add_driver(d1)
system.add_driver(d2)

r1 = Rider("Alice", Location(1, 1))
system.add_rider(r1)

ride = r1.request_ride(system, Location(5, 5))

if ride:
    system.complete_ride(ride, UPI())

Ride started: Alice → Driver1 | Price: ₹56.568542494923804
Paid ₹56.568542494923804 via UPI
Ride completed
